# BigQuant 量化大赛 · 端到端赛道 · 单 notebook 提交

模型包: `cnn_3f`

**内存优化说明（v2）**

为满足 51.2 GB 内存上限，训练阶段做了如下改动：

1. **按年份分块加载**：不再一次性读入 48 个月 bar1m，而是逐年份读取 + 处理后立即释放原始表。
2. **逐块累积样本，避免大数组 2× 峰值**：`_cat_chunks` 预分配目标数组并逐块填充+释放源块。
3. **去 pandas 全量 copy**：`restore_units` / `tag_time` / `_normalize` 就地修改；用整型年月判断替代 `strftime`。
4. **全链路 float32**，弃用 float64 冗余副本。
5. **显式 `gc.collect()`**：每一年处理完后释放 `df1m / tr_df / va_df / per` 等。

推理路径 `main` 未变，仍只读入 start~end 区间（含 10 天回看缓冲），单次几百 MB 级。


In [ ]:
import types
# ======================================================================
# 超参数配置 + 随机种子 (单一事实来源)
# ======================================================================
CFG = types.SimpleNamespace(
    SEED=42,
    FREQS=['bar30m', 'bar15m', 'bar5m'],
    LOOKBACK_MINUTES=120,
    HORIZON_MINUTES=30,
    SAMPLE_STEP_MINUTES=30,
    INPUT_SAME_DAY_ONLY=False,
    TRAIN_MONTHS=['202001', '202002', '202003', '202004', '202005', '202006', '202007', '202008', '202009', '202010', '202011', '202012', '202101', '202102', '202103', '202104', '202105', '202106', '202107', '202108', '202109', '202110', '202111', '202112', '202201', '202202', '202203', '202204', '202205', '202206', '202207', '202208', '202209', '202210', '202211', '202212', '202301', '202302', '202303', '202304', '202305', '202306', '202307', '202308', '202309', '202310', '202311', '202312'],
    MODEL_CFG={'in_dim': 25, 'hidden': 64, 'conv_blocks': 3, 'kernel': 3, 'transformer_layers': 2, 'heads': 4, 'dropout': 0.2, 'fused_dim': 64},
    TRAIN_CFG={'epochs': 12, 'lr': 0.001, 'weight_decay': 1e-05, 'batch_size': 4096, 'patience': 4, 'grad_clip': 1.0, 'val_months': 2, 'lr_factor': 0.5, 'lr_patience': 2, 'loss': 'corr', 'target': 'rank'},
)


In [ ]:
"""
# Transformer 端到端 —— 训练侧 (单一事实来源)

本文件同时承担两件事, 是训练与推理的**唯一事实来源**:

1. **本地阶段**: 在写死的训练区间上从零训练 CNN+Transformer, 把
   **权重 + 标准化统计 + 结构超参** 存到 `transformer_model.json`。
2. **私榜重训阶段**: 平台在隔离环境调用 `train_and_save(datasources)` 从零重训。

推理脚本从本文件 import 共享定义, 不重复实现, 保证 train/infer 预处理严格一致。

## 关键设计

- 只依赖 1 分钟 K 线 (`cpt_jyc_2026_e2e_bar1m`), 内部重采样到 5/15/30 分钟。
- CNN 滑窗 + Transformer: 1x1 Conv 投影 -> 多个膨胀卷积残差块 -> 正弦位置编码
  -> TransformerEncoder -> 注意力池化‖平均池化 -> MLP head。
- 多频融合: 每个频率一条独立编码分支, 池化后拼接, 残差式 head 输出标量。
- 标准化统计随权重落盘: 训练集 fit, 推理直接复用。
- 不做任何手工因子, 只做单位还原、量纲统一、对数变换与缺失值处理。

## 内存优化 (v2)

- 训练数据**按年分块**读入/处理/释放, 单年峰值大幅下降。
- 大数组 (样本 X) 使用 `_cat_chunks` **逐块填充**, 避免 `np.concatenate` 2× 峰值。
- pandas 侧只保留必要列, 全链路 float32, `del` + `gc.collect()` 及时回收。
"""
import gc
import json
import math
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
# ==========================================================================
# 常量
# ==========================================================================
import config as CFG            # 超参数 + 随机种子 (单一事实来源)

MODEL_PATH = "transformer_model.json"
BATCH = CFG.TRAIN_CFG["batch_size"]
SEED = CFG.SEED

FREQ_MINUTES = {"bar1m": 1, "bar5m": 5, "bar15m": 15, "bar30m": 30}
MORNING_START, MORNING_END = 570, 690      # 09:30, 11:30
AFTERNOON_START, AFTERNOON_END = 780, 900  # 13:00, 15:00
AUCTION_MINUTE = 565                       # 09:25 集合竞价截面
SAMPLE_STEP = CFG.SAMPLE_STEP_MINUTES
HORIZON_MINUTES = CFG.HORIZON_MINUTES

ALIGNED_MINUTES = [AUCTION_MINUTE] + [
    m for m in range(MORNING_START + SAMPLE_STEP, MORNING_END + 1, SAMPLE_STEP)
] + [
    m for m in range(AFTERNOON_START + SAMPLE_STEP, AFTERNOON_END + 1, SAMPLE_STEP)
]

_PRICE_FIELDS = ["open", "high", "low", "close", "amount",
                 "ask_price1", "ask_price2", "ask_price3",
                 "bid_price1", "bid_price2", "bid_price3"]
_OHLC = ["open", "high", "low", "close"]
_BOOK = [f"{s}_{k}{i}" for s in ("ask", "bid")
         for k in ("price", "volume", "num_orders") for i in (1, 2, 3)]

KLINE_FEATURES = ["open_rel", "high_rel", "low_rel", "close_ret",
                  "log_volume", "log_amount", "log_deal_number"]
ORDERBOOK_FEATURES = (
    [f"{s}_price{i}_rel" for s in ("ask", "bid") for i in (1, 2, 3)]
    + [f"log_{s}_volume{i}" for s in ("ask", "bid") for i in (1, 2, 3)]
    + [f"log_{s}_orders{i}" for s in ("ask", "bid") for i in (1, 2, 3)]
)
ALL_FEATURES = KLINE_FEATURES + ORDERBOOK_FEATURES
N_FEATURES = len(ALL_FEATURES)  # 25


In [ ]:
# ==========================================================================
# 模型 (与本地训练代码结构逐位一致)
# ==========================================================================
class PositionalEncoding(nn.Module):
    """正弦位置编码(非学习式, 泛化到任意长度)。"""

    def __init__(self, dim: int, max_len: int = 500):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class ConvResBlock(nn.Module):
    """1D 卷积残差块 (Pre-LN + GELU)。输入/输出 [B, dim, T]。"""

    def __init__(self, dim, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()
        pad = (kernel_size - 1) // 2 * dilation
        self.conv1 = nn.Conv1d(dim, dim, kernel_size, padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(dim, dim, kernel_size, padding=pad, dilation=dilation)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = self.norm1(x.transpose(1, 2)).transpose(1, 2)
        x = F.gelu(x)
        x = self.conv1(x)
        x = self.norm2(x.transpose(1, 2)).transpose(1, 2)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.conv2(x)
        return x + residual


class AttentionPooling(nn.Module):
    """注意力池化 + 平均池化拼接: [B,T,D] -> [B,2D]。"""

    def __init__(self, dim):
        super().__init__()
        h = max(dim // 2, 8)
        self.score = nn.Sequential(nn.Linear(dim, h), nn.Tanh(), nn.Linear(h, 1))
        self.out_dim = dim * 2

    def forward(self, x):
        w = torch.softmax(self.score(x), dim=1)
        att = (x * w).sum(dim=1)
        avg = x.mean(dim=1)
        return torch.cat([att, avg], dim=-1)


class CNNTransformerEncoder(nn.Module):
    """CNN + Transformer 编码器: [B,T,in_dim] -> [B,T,hidden]。"""

    def __init__(self, in_dim=25, hidden_dim=64, conv_blocks=3, kernel_size=3,
                 dilation_growth=1, transformer_layers=2, n_heads=4, dropout=0.1):
        super().__init__()
        self.stem = nn.Conv1d(in_dim, hidden_dim, 1)
        dilations = [dilation_growth ** i for i in range(conv_blocks)]
        self.conv_blocks = nn.ModuleList(
            [ConvResBlock(hidden_dim, kernel_size, d, dropout) for d in dilations])
        self.pos_enc = PositionalEncoding(hidden_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=n_heads, dim_feedforward=hidden_dim * 4,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=transformer_layers)
        self.out_norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.stem(x)
        for block in self.conv_blocks:
            x = block(x)
        x = x.transpose(1, 2)
        x = self.pos_enc(x)
        x = self.transformer(x)
        return self.out_norm(x)


class StockTransformer(nn.Module):
    """单频模型: Encoder -> Pool -> Head -> scalar。"""

    def __init__(self, in_dim=N_FEATURES, hidden=64, conv_blocks=3, kernel=3,
                 transformer_layers=2, heads=4, dropout=0.1, fused_dim=64):
        super().__init__()
        self.encoder = CNNTransformerEncoder(
            in_dim, hidden, conv_blocks, kernel, 1, transformer_layers, heads, dropout)
        self.pool = AttentionPooling(hidden)
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(self.pool.out_dim, fused_dim),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(fused_dim, 1))

    def forward(self, x):
        if isinstance(x, (list, tuple)):
            x = x[0]
        return self.head(self.pool(self.encoder(x))).squeeze(-1)


class StockTransformerMF(nn.Module):
    """多频模型: 每频一条 CNN+Transformer 分支, 拼接后残差 head。"""

    def __init__(self, freqs, in_dim=N_FEATURES, hidden=64, conv_blocks=3, kernel=3,
                 transformer_layers=2, heads=4, dropout=0.1, fused_dim=64):
        super().__init__()
        self.freqs = list(freqs)
        self.branch = nn.ModuleDict()
        self.pool = nn.ModuleDict()
        for f in self.freqs:
            self.branch[f] = CNNTransformerEncoder(
                in_dim, hidden, conv_blocks, kernel, 1, transformer_layers, heads, dropout)
            self.pool[f] = AttentionPooling(hidden)
        d = len(self.freqs) * 2 * hidden
        self.fuse = nn.Sequential(nn.Linear(d, fused_dim), nn.ReLU(),
                                  nn.LayerNorm(fused_dim))
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(fused_dim + d, fused_dim),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(fused_dim, 1))

    def forward(self, xs):
        parts = [self.pool[f](self.branch[f](xs[k])) for k, f in enumerate(self.freqs)]
        h = torch.cat(parts, dim=-1)
        return self.head(torch.cat([self.fuse(h), h], dim=-1)).squeeze(-1)


def build_model(model_cfg):
    """按 model_cfg 构造模型(arch='single'|'multi')。"""
    kw = {k: v for k, v in model_cfg.items() if k not in ("arch", "freqs")}
    if model_cfg["arch"] == "multi":
        return StockTransformerMF(freqs=model_cfg["freqs"], **kw)
    return StockTransformer(**kw)


In [ ]:
# ==========================================================================
# 数据: 读表 -> 还原 -> 重采样 -> 特征 -> 切窗
# 内存: 全程尽量就地修改, 不做整表 .copy()
# ==========================================================================
def _fetch_table(table, start, end):
    """
    取数据为 DataFrame。
    - DataFrame: 直接使用;
    - 平台表对象: 经 to_pandas()/to_df()/df() 取出;
    - 表名(str): 走 dai.query, **必须带日期过滤**; 多取 10 天回看缓冲。
    """
    if isinstance(table, pd.DataFrame):
        return table
    for attr in ("to_pandas", "to_df"):
        if hasattr(table, attr):
            obj = getattr(table, attr)
            return obj() if callable(obj) else obj
    if hasattr(table, "df"):
        obj = table.df
        return obj() if callable(obj) else obj
    import dai
    buf = (pd.Timestamp(start) - pd.Timedelta(days=10)).strftime("%Y-%m-%d")
    end_s = pd.Timestamp(end).strftime("%Y-%m-%d %H:%M:%S")
    return dai.query(f"SELECT * FROM {table}",
                     filters={"date": [buf, end_s]}).df()


def _to_dataframe(table, start=None, end=None):
    return _fetch_table(table, start, end)


def _normalize(df):
    """归一列名: 时间列 -> date, 标的列 -> instrument_id (尽量就地)。"""
    ren = {}
    for c in df.columns:
        lc = c.lower()
        if lc in ("time", "datetime", "date"):
            ren[c] = "date"
        elif lc in ("instrument", "symbol", "stock", "code", "sid"):
            ren[c] = "instrument_id"
    if ren:
        df = df.rename(columns=ren)
    if not pd.api.types.is_datetime64_any_dtype(df["date"]):
        df["date"] = pd.to_datetime(df["date"])
    return df


def restore_units(df):
    """单位还原: OHLC 的 -1 -> NaN 并 /100; amount/盘口价格 /100;
    盘口价格为 0 表示该侧无挂单 -> NaN。全程 float32, 就地修改。"""
    for f in _OHLC:
        if f in df.columns:
            df[f] = df[f].astype("float32", copy=False)
            df.loc[df[f] == -1, f] = np.nan
            df[f] = (df[f] / np.float32(100.0)).astype("float32", copy=False)
    for f in _PRICE_FIELDS:
        if f in df.columns and f not in _OHLC:
            df[f] = (df[f].astype("float32", copy=False) / np.float32(100.0)).astype("float32", copy=False)
    for f in _BOOK:
        if f in df.columns and f.endswith(("1", "2", "3")) and "price" in f:
            df.loc[df[f] == 0.0, f] = np.nan
    return df


def tag_time(df):
    """打时间标签: minute_of_day; 剔除 09:25 之前的盘前指示价 bar(保留 09:25)。"""
    mod = (df["date"].dt.hour * 60 + df["date"].dt.minute).to_numpy(dtype="int32")
    keep = mod >= AUCTION_MINUTE
    out = df[keep]                                   # 布尔索引, 只保留必要行
    if len(out) == 0:
        out = out.copy()
        out["minute_of_day"] = np.zeros(0, dtype="int16")
        out["day"] = pd.Series([], dtype="datetime64[ns]")
        return out.reset_index(drop=True)
    out = out.copy()
    out["minute_of_day"] = mod[keep].astype("int16")
    out["day"] = out["date"].dt.normalize()
    return out.reset_index(drop=True)


def resample_bars(df1m, kmins):
    """把 1 分钟 bar 聚合成 kmins 分钟 bar。"""
    df = df1m.sort_values(["instrument_id", "date"], kind="mergesort")
    auction = df[df["minute_of_day"] == AUCTION_MINUTE].copy()
    cont = df[df["minute_of_day"] > AUCTION_MINUTE].copy()
    del df

    rel = cont["minute_of_day"].to_numpy() - MORNING_START
    cont["_end"] = MORNING_START + np.ceil(rel / kmins).astype("int64") * kmins

    agg = {"open": ("open", "first"), "high": ("high", "max"),
           "low": ("low", "min"), "close": ("close", "last"),
           "volume": ("volume", "sum"), "amount": ("amount", "sum"),
           "deal_number": ("deal_number", "sum")}
    for c in _BOOK:
        agg[c] = (c, "last")

    g = (cont.groupby(["instrument_id", "day", "_end"], sort=False)
             .agg(**agg).reset_index()
             .rename(columns={"_end": "minute_of_day"}))
    del cont

    cols = ["instrument_id", "minute_of_day", "day"] + list(agg.keys())
    a = auction[[c for c in cols if c in auction.columns]]
    out = pd.concat([a, g[[c for c in cols if c in g.columns]]], ignore_index=True)
    del auction, g
    out["date"] = (pd.to_datetime(out["day"])
                   + pd.to_timedelta(out["minute_of_day"], unit="m"))
    return (out.dropna(subset=["day"])
               .sort_values(["instrument_id", "date"])
               .reset_index(drop=True))


def build_features(df):
    """构造 25 个特征(只做单位/量纲统一与对数变换)。"""
    close = df["close"].to_numpy(dtype="float32")
    valid = close > 0
    out = pd.DataFrame(index=df.index)
    inv = np.where(valid, 1.0 / np.where(valid, close, 1.0), np.nan).astype("float32")
    out["open_rel"] = (df["open"].to_numpy(dtype="float32") * inv - 1.0).astype("float32")
    out["high_rel"] = (df["high"].to_numpy(dtype="float32") * inv - 1.0).astype("float32")
    out["low_rel"]  = (df["low"].to_numpy(dtype="float32") * inv - 1.0).astype("float32")
    prev = df.groupby(["instrument_id", "day"], sort=False)["close"].shift(1).to_numpy(dtype="float32")
    prv_ok = prev > 0
    out["close_ret"] = np.where(prv_ok, close / np.where(prv_ok, prev, 1.0) - 1.0, np.nan).astype("float32")
    out["log_volume"] = np.log1p(df["volume"].clip(lower=0).to_numpy(dtype="float32")).astype("float32")
    out["log_amount"] = np.log1p(df["amount"].clip(lower=0).to_numpy(dtype="float32")).astype("float32")
    out["log_deal_number"] = np.log1p(df["deal_number"].clip(lower=0).to_numpy(dtype="float32")).astype("float32")
    for i in (1, 2, 3):
        for s in ("ask", "bid"):
            out[f"{s}_price{i}_rel"] = (df[f"{s}_price{i}"].to_numpy(dtype="float32") * inv - 1.0).astype("float32")
            out[f"log_{s}_volume{i}"] = np.log1p(df[f"{s}_volume{i}"].clip(lower=0).to_numpy(dtype="float32")).astype("float32")
            out[f"log_{s}_orders{i}"] = np.log1p(df[f"{s}_num_orders{i}"].clip(lower=0).to_numpy(dtype="float32")).astype("float32")
    return out[ALL_FEATURES]


def build_labels(df, horizon_bars):
    """标签 = future_vwap / close[t+h] - 1, 窗口不跨交易日。"""
    grp = [df["instrument_id"], df["day"]]
    amt64 = df["amount"].astype("float64")
    vol64 = df["volume"].astype("float64")
    c_amt = amt64.groupby(grp, sort=False).cumsum()
    c_vol = vol64.groupby(grp, sort=False).cumsum()
    fwd_amt = c_amt.groupby(grp, sort=False).shift(-horizon_bars) - c_amt
    fwd_vol = c_vol.groupby(grp, sort=False).shift(-horizon_bars) - c_vol
    fwd_vwap = np.where(fwd_vol > 0, fwd_amt / fwd_vol, np.nan)
    end = df["close"].astype("float64").groupby(grp, sort=False).shift(-horizon_bars)
    close = df["close"].to_numpy(dtype="float64")
    end_a = end.to_numpy(dtype="float64")
    lab = np.where((close > 0) & (end_a > 0), fwd_vwap / np.where(end_a > 0, end_a, 1.0) - 1.0, np.nan)
    return lab.astype("float32")


def make_samples(feat, label, df, seq_len, minutes, with_label=True):
    """向量化滑窗: 返回 (X[N,T,F], y[N], keys[N,3]=inst,day_ord,mod)。"""
    n = len(df)
    if n <= seq_len:
        return (np.zeros((0, seq_len, N_FEATURES), "float32"),
                np.zeros((0,), "float32"), np.zeros((0, 3), "int64"))
    inst = df["instrument_id"].to_numpy()
    mod = df["minute_of_day"].to_numpy()
    day = pd.to_datetime(df["day"]).to_numpy().astype("datetime64[D]").astype("int64")

    i = np.arange(seq_len, n)
    keep = np.isin(mod[i], np.asarray(sorted(set(minutes)), dtype=mod.dtype))
    keep &= (inst[i] == inst[i - seq_len])
    if with_label:
        keep &= ~np.isnan(label[i])
    sel = i[keep]
    if sel.size == 0:
        return (np.zeros((0, seq_len, N_FEATURES), "float32"),
                np.zeros((0,), "float32"), np.zeros((0, 3), "int64"))
    win = sel[:, None] + np.arange(-seq_len, 0)[None, :]
    X = feat[win].astype("float32", copy=False)
    y = label[sel].astype("float32", copy=False) if with_label else np.zeros(len(sel), "float32")
    keys = np.stack([inst[sel], day[sel].astype("int64"), mod[sel]], axis=1)
    return X, y, keys


class _FreqData:
    """单频: 构造特征 + (可选)标签 + 滑窗。"""

    def __init__(self, df1m, freq, seq_len, minutes, with_label=True):
        self.freq = freq
        self.seq_len = seq_len
        km = FREQ_MINUTES[freq]
        bars = resample_bars(df1m, km)
        feat = build_features(bars).to_numpy(dtype="float32")
        lab = build_labels(bars, max(1, HORIZON_MINUTES // km)) if with_label \
            else np.full(len(bars), np.nan, "float32")
        self.X, self.y, self.keys = make_samples(feat, lab, bars.reset_index(drop=True),
                                                 seq_len, minutes, with_label)
        del bars, feat, lab


def _subset_freq(fd, sel):
    """按行下标取 _FreqData 的子集。"""
    o = _FreqData.__new__(_FreqData)
    o.freq, o.seq_len = fd.freq, fd.seq_len
    o.X = fd.X[sel]
    o.y = fd.y[sel]
    o.keys = fd.keys[sel]
    return o


def _fit_scaler(X, chunk=200_000):
    """训练集特征均值/标准差(分块累加, 忽略 NaN)。"""
    nf = X.shape[-1]
    s = np.zeros(nf, "float64"); sq = np.zeros(nf, "float64"); c = np.zeros(nf, "float64")
    for st in range(0, len(X), chunk):
        blk = X[st:st + chunk].reshape(-1, nf).astype("float64")
        m = ~np.isnan(blk)
        z = np.where(m, blk, 0.0)
        s += z.sum(0); sq += (z * z).sum(0); c += m.sum(0)
    c = np.maximum(c, 1.0)
    mean = s / c
    var = np.maximum(sq / c - mean * mean, 0.0)
    std = np.sqrt(var)
    mean = np.where(np.isnan(mean), 0.0, mean)
    std = np.where((np.isnan(std)) | (std < 1e-6), 1.0, std)
    return mean.astype("float32"), std.astype("float32")


def _apply_scaler(X, mean, std):
    """逐样本标准化, NaN -> 0。注意输入需为可写的连续 float32 (由 make_samples 保证)。"""
    out = X if X.dtype == np.float32 else X.astype("float32")
    mean = np.asarray(mean, "float32").reshape(1, 1, -1)
    std = np.asarray(std, "float32").reshape(1, 1, -1)
    chunk = max(1, 2_000_000 // max(out.shape[1], 1))
    for st in range(0, len(out), chunk):
        blk = out[st:st + chunk]
        np.subtract(blk, mean, out=blk)
        np.divide(blk, std, out=blk)
        np.nan_to_num(blk, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return out


def _rank_targets(y, keys):
    """截面 rank 目标: 同一 (day, mod) 内做百分位排名并映射到 [-1,1]。"""
    out = np.zeros_like(y, "float32")
    df = pd.DataFrame({"y": y, "day": keys[:, 1], "mod": keys[:, 2]})
    for _, g in df.groupby(["day", "mod"], sort=False):
        if len(g) < 2:
            continue
        r = g["y"].rank(method="average")
        out[g.index.to_numpy()] = ((r / len(g)) * 2.0 - 1.0).to_numpy("float32")
    return out


def clip_bounds(labels, lo=0.01, hi=0.99):
    x = labels[np.isfinite(labels)]
    if x.size == 0:
        return -np.inf, np.inf
    a, b = np.quantile(x, [lo, hi])
    return float(a), float(b)


In [ ]:
# ==========================================================================
# 推理侧统一入口: build_dataset (训练/推理共用, 保证口径一致)
# ==========================================================================
def build_dataset(table, start_date, end_date, mode, minutes, stats,
                  freqs=None, seq_lens=None):
    """从 1 分钟表构造模型输入。"""
    freqs = freqs or ["bar30m"]
    seq_lens = seq_lens or {}
    if minutes is None:
        minutes = ALIGNED_MINUTES

    df = _normalize(_fetch_table(table, start_date, end_date))
    df = restore_units(df)
    df = tag_time(df)

    with_label = (mode != "infer")
    per = {}
    for f in freqs:
        T = int(seq_lens.get(f, max(1, 120 // FREQ_MINUTES[f])))
        per[f] = _FreqData(df, f, T, minutes, with_label=with_label)
    del df
    gc.collect()

    d0 = np.datetime64(pd.Timestamp(start_date).date(), "D").astype("int64")
    d1 = np.datetime64(pd.Timestamp(end_date).date(), "D").astype("int64")
    for f in freqs:
        k = per[f].keys
        m = (k[:, 1] >= d0) & (k[:, 1] <= d1)
        per[f] = _subset_freq(per[f], np.where(m)[0])

    if len(freqs) > 1:
        sets = [set(map(tuple, per[f].keys.tolist())) for f in freqs]
        common = set.intersection(*sets)
        common = set(sorted(common))
        idx = {}
        for f in freqs:
            mask = np.fromiter((tuple(r) in common for r in per[f].keys.tolist()),
                               dtype=bool, count=len(per[f].keys))
            idx[f] = np.where(mask)[0]
    else:
        idx = {freqs[0]: np.arange(len(per[freqs[0]].keys))}

    ref = freqs[0]
    keys = per[ref].keys[idx[ref]]
    y = per[ref].y[idx[ref]]

    X_list = []
    for f in freqs:
        X = per[f].X[idx[f]]
        mean, std = stats[f]
        X_list.append(_apply_scaler(X, mean, std))
    del per
    gc.collect()

    idx_df = pd.DataFrame({
        "date": (pd.to_datetime(keys[:, 1], unit="D")
                 + pd.to_timedelta(keys[:, 2], unit="m")),
        "instrument": keys[:, 0].astype("int64"),
        "minute_of_day": keys[:, 2].astype("int64"),
    })
    return X_list, y, idx_df, keys


In [ ]:
# ==========================================================================
# 模型存取 (JSON: 张量按 dtype/shape/list 序列化)
# ==========================================================================
def _enc(obj):
    if isinstance(obj, dict):
        return {k: _enc(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_enc(v) for v in obj]
    if isinstance(obj, torch.Tensor):
        a = obj.detach().cpu().numpy()
        return {"__t__": 1, "dtype": str(a.dtype), "shape": list(a.shape),
                "data": a.reshape(-1).tolist()}
    if isinstance(obj, np.ndarray):
        return {"__t__": 1, "dtype": str(obj.dtype), "shape": list(obj.shape),
                "data": obj.reshape(-1).tolist()}
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    return obj


def _dec(obj):
    if isinstance(obj, dict):
        if obj.get("__t__"):
            a = np.array(obj["data"], dtype=obj["dtype"]).reshape(obj["shape"])
            return a
        return {k: _dec(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_dec(v) for v in obj]
    return obj


def save_model(path, payload):
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(_enc(payload), fh)


def load_model(path, map_location="cpu"):
    with open(path, "r", encoding="utf-8") as fh:
        raw = json.load(fh)
    ck = _dec(raw)
    if "state_dict" in ck:
        ck["state_dict"] = {k: torch.as_tensor(v, device=map_location)
                            for k, v in ck["state_dict"].items()}
    return ck


In [ ]:
# ==========================================================================
# 数据加载 / 分块累积辅助
# ==========================================================================
def _load_range(table, months_or_start, end=None):
    """按月份列表或日期区间读 1 分钟表, 返回还原+打标后的 DataFrame。

    内存: 用整数 (year*100+month) 判月份, 避免 strftime 全量 str 副本。
    """
    if end is None:
        months = [str(m) for m in months_or_start]
        start_s = f"{months[0][:4]}-{months[0][4:6]}-01"
        last = months[-1]
        end_s = (pd.Timestamp(f"{last[:4]}-{last[4:6]}-01")
                 + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d 23:59:59")
        df = _fetch_table(table, start_s, end_s)
        df = _normalize(df)
        df = restore_units(df)
        ym = (df["date"].dt.year * 100 + df["date"].dt.month).to_numpy(dtype="int64")
        keep = np.isin(ym, np.array([int(m) for m in months], dtype="int64"))
        df = df.loc[keep].reset_index(drop=True)
        del ym, keep
    else:
        df = _fetch_table(table, months_or_start, end)
        df = _normalize(df)
        df = restore_units(df)
        m = (df["date"] >= pd.Timestamp(months_or_start)) & (df["date"] <= pd.Timestamp(end))
        df = df.loc[m].reset_index(drop=True)
        del m
    df = tag_time(df)
    return df


def _align_per(per, freqs):
    """多频: 按 keys 取交集, 返回每频在各自 X 中的行下标。"""
    if len(freqs) == 1:
        f = freqs[0]
        return {f: np.arange(len(per[f].keys))}
    sets = [set(map(tuple, per[f].keys.tolist())) for f in freqs]
    common = set(sorted(set.intersection(*sets)))
    out = {}
    for f in freqs:
        mask = np.fromiter((tuple(r) in common for r in per[f].keys.tolist()),
                           dtype=bool, count=len(per[f].keys))
        out[f] = np.where(mask)[0]
    return out


def _cat_chunks(chunk_dict):
    """逐块拼接, 每处理完一块立即置 None 释放引用, 避免 np.concatenate 2× 峰值。
    chunk_dict: {"X": [arr1, arr2, ...], "y": [...], "keys": [...]}
    """
    out = {}
    for key, chunks in chunk_dict.items():
        chunks = [c for c in chunks if c is not None and len(c) > 0]
        if not chunks:
            out[key] = np.zeros((0,), dtype="float32")
            continue
        total = sum(c.shape[0] for c in chunks)
        shape = (total,) + chunks[0].shape[1:]
        arr = np.empty(shape, dtype=chunks[0].dtype)
        pos = 0
        for i in range(len(chunks)):
            c = chunks[i]
            n = c.shape[0]
            arr[pos:pos + n] = c
            pos += n
            chunks[i] = None          # 允许 GC
        out[key] = arr
        del chunks
    return out


In [ ]:
# ==========================================================================
# 训练循环 (与旧 _train_freqs 保持同等语义, 只拆出数据准备部分)
# ==========================================================================
@torch.no_grad()
def _eval_ic(model, X, y, keys, device, batch=4096):
    model.eval()
    Xt = [torch.from_numpy(np.ascontiguousarray(x)) for x in X]
    preds = []
    for s in range(0, len(y), batch):
        xb = [t[s:s + batch].to(device) for t in Xt]
        preds.append(model(xb).cpu().numpy())
    preds = np.concatenate(preds) if preds else np.zeros(0)
    m = np.isfinite(y)
    if m.sum() < 10:
        return 0.0
    df = pd.DataFrame({"p": preds[m], "y": y[m], "day": keys[m, 1], "mod": keys[m, 2]})
    ics = []
    for _, g in df.groupby(["day", "mod"], sort=False):
        if len(g) < 10:
            continue
        pr = g["p"].rank(); yr = g["y"].rank()
        ics.append(np.corrcoef(pr, yr)[0, 1])
    return float(np.mean(ics)) if ics else 0.0


def _train_loop(Xtr_list, ytr, Xva_list, yva_raw, vkeys, model_cfg, seed=SEED,
                epochs=None, lr=None, batch=None, patience=None, weight_decay=None):
    """核心训练循环, 直接吃预处理好的 numpy 分频数组。"""
    tc = CFG.TRAIN_CFG
    epochs = tc["epochs"] if epochs is None else epochs
    lr = tc["lr"] if lr is None else lr
    batch = tc["batch_size"] if batch is None else batch
    patience = tc["patience"] if patience is None else patience
    weight_decay = tc["weight_decay"] if weight_decay is None else weight_decay

    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(model_cfg).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, factor=tc["lr_factor"], patience=tc["lr_patience"])

    Xtr_t = [torch.from_numpy(np.ascontiguousarray(x)) for x in Xtr_list]
    ytr_t = torch.from_numpy(ytr)
    n = len(ytr_t)
    best, best_state, bad = -9e9, None, 0

    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        tot = 0.0; nb = 0
        for s in range(0, n, batch):
            idxb = perm[s:s + batch]
            xb = [t[idxb].to(device) for t in Xtr_t]
            yb = ytr_t[idxb].to(device)
            opt.zero_grad()
            p = model(xb)
            pc = p - p.mean(); yc = yb - yb.mean()
            corr = (pc * yc).sum() / (torch.sqrt((pc ** 2).sum() * (yc ** 2).sum()) + 1e-8)
            loss = 1.0 - corr
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += float(loss); nb += 1

        ic = _eval_ic(model, Xva_list, yva_raw, vkeys, device)
        sched.step(-ic)
        print(f"  ep{ep+1:02d} loss={tot/max(nb,1):.4f} val_IC={ic:+.4f}")
        if ic > best:
            best, bad = ic, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    print(f"  best val_IC={best:+.4f}")
    return model


In [ ]:
# ==========================================================================
# 训练 + 保存 (按年分块, 峰值内存受控)
# ==========================================================================
def train_and_save(datasources, train_months=None, freqs=None, model_path=MODEL_PATH):
    """从零训练并保存 transformer_model.json (私榜重训亦调用本函数)。

    内存策略: 按年份分块读取 bar1m -> 分块处理成样本 -> 分块累积 -> 逐块拼接。
    每年处理完毕立即 del + gc.collect(), 单次驻留的原始表只有一年的量级。
    """
    if train_months is None:
        train_months = CFG.TRAIN_MONTHS
    if freqs is None:
        freqs = list(CFG.FREQS)
    freqs = list(freqs)

    table = datasources["bar1m"]
    seq_lens = {f: max(1, CFG.LOOKBACK_MINUTES // FREQ_MINUTES[f]) for f in freqs}

    months_sorted = sorted([str(m) for m in train_months])
    val_n = int(CFG.TRAIN_CFG["val_months"])
    val_months_set = set(months_sorted[-val_n:]) if len(months_sorted) > val_n + 1 else set(months_sorted[-1:])
    val_ints = np.array(sorted(int(m) for m in val_months_set), dtype="int64")

    years = sorted(set(m[:4] for m in months_sorted))
    print(f"[train] 训练区间 {months_sorted[0]} ~ {months_sorted[-1]}, "
          f"共 {len(years)} 年, seed={SEED}")

    tr_chunks = {f: {"X": [], "y": [], "keys": []} for f in freqs}
    va_chunks = {f: {"X": [], "y": [], "keys": []} for f in freqs}

    for year in years:
        year_months = sorted(m for m in months_sorted if m.startswith(year))
        print(f"[train] [{year}] 读取 {len(year_months)} 个月 bar1m ...")
        df1m = _load_range(table, year_months)
        if len(df1m) == 0:
            print(f"[train] [{year}] 无数据, 跳过")
            del df1m; gc.collect()
            continue
        n_days = df1m["day"].nunique()
        print(f"[train] [{year}] rows={len(df1m):,} days={n_days}")

        # 拆分 train / val (不产生整表副本)
        ym = (df1m["date"].dt.year * 100 + df1m["date"].dt.month).to_numpy(dtype="int64")
        is_val = np.isin(ym, val_ints)
        del ym
        tr_df = df1m.loc[~is_val].reset_index(drop=True)
        va_df = df1m.loc[is_val].reset_index(drop=True)
        del df1m, is_val
        gc.collect()

        # 每频构造样本
        tr_per = {f: _FreqData(tr_df, f, seq_lens[f], ALIGNED_MINUTES, True) for f in freqs}
        va_per = {f: _FreqData(va_df, f, seq_lens[f], ALIGNED_MINUTES, True) for f in freqs}
        del tr_df, va_df
        gc.collect()

        tr_idx = _align_per(tr_per, freqs)
        va_idx = _align_per(va_per, freqs)

        for f in freqs:
            tr_chunks[f]["X"].append(tr_per[f].X[tr_idx[f]])
            tr_chunks[f]["y"].append(tr_per[f].y[tr_idx[f]])
            tr_chunks[f]["keys"].append(tr_per[f].keys[tr_idx[f]])
            va_chunks[f]["X"].append(va_per[f].X[va_idx[f]])
            va_chunks[f]["y"].append(va_per[f].y[va_idx[f]])
            va_chunks[f]["keys"].append(va_per[f].keys[va_idx[f]])
        del tr_per, va_per, tr_idx, va_idx
        gc.collect()

    # 逐块拼接 (预分配 + 逐块释放)
    print("[train] 拼接分块 ...")
    tr_data = {f: _cat_chunks(tr_chunks[f]) for f in freqs}
    va_data = {f: _cat_chunks(va_chunks[f]) for f in freqs}
    del tr_chunks, va_chunks
    gc.collect()
    for f in freqs:
        print(f"[train]   {f}: train={len(tr_data[f]['X']):,}  val={len(va_data[f]['X']):,}")

    # 标准化
    print("[train] 拟合标准化统计 ...")
    stats = {f: _fit_scaler(tr_data[f]["X"]) for f in freqs}
    for f in freqs:
        tr_data[f]["X"] = _apply_scaler(tr_data[f]["X"], *stats[f])
        va_data[f]["X"] = _apply_scaler(va_data[f]["X"], *stats[f])
    gc.collect()

    # 目标
    ref_f = freqs[0]
    ytr = _rank_targets(tr_data[ref_f]["y"], tr_data[ref_f]["keys"])
    yva_raw = va_data[ref_f]["y"]
    vkeys = va_data[ref_f]["keys"]

    model_cfg = dict(arch="multi" if len(freqs) > 1 else "single",
                     freqs=list(freqs), **CFG.MODEL_CFG)

    print("[train] 开始训练 ...")
    model = _train_loop(
        [tr_data[f]["X"] for f in freqs], ytr,
        [va_data[f]["X"] for f in freqs], yva_raw, vkeys,
        model_cfg, seed=SEED)

    payload = {
        "format": "miniquant-transformer-v1",
        "arch": model_cfg["arch"],
        "freqs": list(freqs),
        "seq_lens": {f: int(seq_lens[f]) for f in freqs},
        "model_cfg": model_cfg,
        "mean": {f: stats[f][0] for f in freqs},
        "std": {f: stats[f][1] for f in freqs},
        "features": ALL_FEATURES,
        "aligned_minutes": ALIGNED_MINUTES,
        "train_months": list(train_months),
        "seed": int(SEED),
        "train_cfg": dict(CFG.TRAIN_CFG),
        "lookback_minutes": int(CFG.LOOKBACK_MINUTES),
        "horizon_minutes": int(CFG.HORIZON_MINUTES),
        "sort_key": 1,
        "state_dict": {k: v for k, v in model.state_dict().items()},
    }
    save_model(model_path, payload)
    print(f"[train] 已保存 {model_path} ({os.path.getsize(model_path)/1e6:.2f} MB)")
    return model_path


In [ ]:
# ==========================================================================
# 推理入口 main
# ==========================================================================
import structlog

logger = structlog.get_logger()


def main(datasources, start_date, end_date):
    """加载已训练好的模型, 在样本外测试区间 (start_date~end_date) 上推理打分。"""
    table = datasources["bar1m"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"未找到模型文件 {MODEL_PATH}; 请先运行 train_and_save(...) 训练并保存, "
            f"再随 notebook 一起上传")

    ckpt = load_model(MODEL_PATH, map_location=device)
    freqs = ckpt["freqs"]
    seq_lens = ckpt["seq_lens"]
    stats = {f: (np.asarray(ckpt["mean"][f], np.float32),
                 np.asarray(ckpt["std"][f], np.float32)) for f in freqs}
    model = build_model(ckpt["model_cfg"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    logger.info("已加载模型", path=MODEL_PATH, freqs=freqs, device=str(device))

    logger.info("构建测试集并预测", start=str(start_date), end=str(end_date))
    Xs, _, idx_df, _ = build_dataset(
        table, start_date, end_date, "infer",
        ckpt["aligned_minutes"], stats, freqs=freqs, seq_lens=seq_lens)

    Xt = [torch.from_numpy(np.ascontiguousarray(x)) for x in Xs]
    preds = []
    with torch.no_grad():
        for i in range(0, len(idx_df), BATCH):
            xb = [t[i:i + BATCH].to(device) for t in Xt]
            preds.append(model(xb).cpu().numpy())
    idx_df["factor"] = (np.concatenate(preds).astype(np.float64)
                        if preds else np.zeros(len(idx_df)))

    import dai
    stk = dai.query("SELECT date, instrument FROM cpt_jyc_2026_instruments",
                    filters={"date": [start_date, end_date]}).df()
    idx_df["date"] = pd.to_datetime(idx_df["date"])
    stk["date"] = pd.to_datetime(stk["date"])
    result = (pd.merge(stk, idx_df, on=["date", "instrument"], how="inner")
                .replace([np.inf, -np.inf], np.nan).dropna(subset=["factor"])
                .drop_duplicates(["date", "instrument"])[["date", "instrument", "factor"]]
                .sort_values(["date", "instrument"])
                .reset_index(drop=True))
    logger.info("因子构建完成", rows=len(result), dates=result["date"].nunique(),
                instruments=result["instrument"].nunique())
    return result


In [ ]:
# ============================================================
# 本地 / 演示入口 (平台公榜/私榜会自行调用 main / train_and_save)
# ============================================================
RUN_LOCAL_DEMO = False          # 本地想看效果改成 True
if RUN_LOCAL_DEMO:
    from bigmodule import M
    datasources = {"bar1m": "cpt_jyc_2026_e2e_bar1m"}
    if not os.path.exists(MODEL_PATH):
        train_and_save(datasources)
    factor_data = main(datasources, "2024-07-01 00:00:00", "2024-09-30 23:59:59")
    print(factor_data.head())
    M.jyc_eval._latest(factor_data=factor_data, show=True)
